In [40]:
# First you need to install the required libraries:

# ARC DATASET

In [41]:
from datasets import load_dataset

ds = load_dataset("allenai/ai2_arc", "ARC-Challenge")

In [42]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})

In [43]:
# Verificar um exemplo do dataset
print("Splits disponíveis:", list(ds.keys()))
print("\n--- Exemplo do split 'train' ---")
example = ds['train'][0]
for key, value in example.items():
    print(f"{key}: {value}")

Splits disponíveis: ['train', 'test', 'validation']

--- Exemplo do split 'train' ---
id: Mercury_SC_415702
question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
choices: {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}
answerKey: A


In [44]:
import os

# Criar pasta datasets
os.makedirs("datasets", exist_ok=True)

# Salvar cada split como Parquet (preserva dicionários e listas)
for split_name, split_data in ds.items():
    filepath = f"datasets/arc_challenge_{split_name}.parquet"
    split_data.to_parquet(filepath)
    print(f"Salvo: {filepath} ({len(split_data)} registros)")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 161.54ba/s]


Salvo: datasets/arc_challenge_train.parquet (1119 registros)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 194.24ba/s]


Salvo: datasets/arc_challenge_test.parquet (1172 registros)


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 216.34ba/s]

Salvo: datasets/arc_challenge_validation.parquet (299 registros)


In [45]:
from datasets import Dataset

# Ler um split do Parquet
train = Dataset.from_parquet("datasets/arc_challenge_train.parquet")
print(train)
print("\n--- Primeiro exemplo ---")
print(train[0])

Generating train split: 1119 examples [00:00, 94564.52 examples/s]

Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 1119
})

--- Primeiro exemplo ---
{'id': 'Mercury_SC_415702', 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'choices': {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'A'}


# Trying to answer questions

In [46]:
train[0]

{'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?',
 'choices': {'text': ['dry palms',
   'wet palms',
   'palms covered with oil',
   'palms covered with lotion'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'A'}

In [47]:
! pip install typing

In [57]:
from typing import TypedDict

def format_sample(sample:TypedDict)->str:
    question = sample['question']
    choices = sample['choices']['text']
    answer_key = sample['answerKey']
    correct_answer = choices[ord(answer_key) - ord('A')]
    
    formatted = f"Question: {question}\nChoices:\n"
    for idx, choice in enumerate(choices):
        formatted += f"{chr(ord('A') + idx)}. {choice}\n"
    formatted += f"Answer: {correct_answer}"
    
    return formatted

def get_question(sample:TypedDict)->str:
    question = sample['question']
    return question

def get_alternatives_label(sample:TypedDict)->str:
    label = sample['choices']['label']
    return label

def get_alternatives_text(sample:TypedDict)->str:
    text = sample['choices']['text']
    return text

def get_correct_answer(sample:TypedDict)->str:
    answer_key = sample['answerKey']
    return answer_key

def get_correct_answer_text(sample:TypedDict)->str:
    correct_answer = get_correct_answer(sample)
    labels = get_alternatives_label(sample)
    index = labels.index(correct_answer)
    text = sample['choices']['text'][index]
    return text

def try_to_answer(sample:TypedDict, guess:str)->bool:
    correct_answer = get_correct_answer(sample).strip().upper() 
    treated_guess = guess.strip().upper()
    if correct_answer == treated_guess:
        return True
    else:
        return False
    
def format_question(sample:TypedDict)->str:
    question = get_question(sample)
    texts = get_alternatives_text(sample)
    labels = get_alternatives_label(sample)

    formatted_choices = "\n".join(
        [f"({label}) {choice}" for label, choice in zip(labels, texts)]
    )
    formatted_question = f"{question}\n{formatted_choices}"

    return formatted_question

In [59]:
print(format_question(sample=train[0]))

George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
(A) dry palms
(B) wet palms
(C) palms covered with oil
(D) palms covered with lotion


In [55]:
train[0]

{'id': 'Mercury_SC_415702',
 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?',
 'choices': {'text': ['dry palms',
   'wet palms',
   'palms covered with oil',
   'palms covered with lotion'],
  'label': ['A', 'B', 'C', 'D']},
 'answerKey': 'A'}